# Vertex AI AutoML Tabular Evaluation & Experiment Tracking

This tutorial demonstrates how to inspect evaluation metrics, extract global feature importance, and track training runs using Vertex AI Experiments with `tabflows`.

### Objectives
1. Retrieve model evaluation metrics (Log Loss, AUC-ROC, PR-AUC) for trained AutoML Tabular models.
2. Extract global feature attributions / importance weights.
3. Track and compare pipeline runs side-by-side using Vertex AI Experiments.

In [ ]:
from dotenv import load_dotenv
from google.cloud import aiplatform

from tabflows import (
    TabularPipelineConfig,
    get_model_evaluation_metrics,
    get_model_feature_attributions,
    list_experiment_runs,
    list_models,
)

# Load environment variables from local .env file
load_dotenv()
print("Environment and libraries loaded successfully.")

In [ ]:
# TabularPipelineConfig automatically loads GCP_PROJECT, GCP_LOCATION, GCP_BUCKET_URI
config = TabularPipelineConfig()

print(f"Project ID: {config.project_id}")
print(f"Location: {config.location}")
print(f"Experiment Name: {config.experiment_name}")

# Discover recent trained models in Vertex AI Model Registry
try:
    models = list_models(config=config, limit=5)
    tabular_models = [
        m
        for m in models
        if "tabular" in m.display_name.lower() or "automl" in m.display_name.lower()
    ]
    target_models = tabular_models if tabular_models else models

    if target_models:
        model = target_models[0]
        print(f"Selected Trained Model: {model.display_name} ({model.resource_name})")
    else:
        print("No trained models found in Vertex AI Model Registry.")
except Exception as e:
    print(f"Could not list models: {e}")

## 1. Model Evaluation Metrics

Inspect classification evaluation metrics generated during model training in Vertex AI Model Registry.

In [ ]:
if "model" in locals() and isinstance(model, aiplatform.Model):
    print(f"Fetching evaluation metrics for model '{model.resource_name}'...")
    metrics = get_model_evaluation_metrics(model=model, config=config)
    print("\n--- Model Evaluation Summary ---")
    for key, val in metrics.items():
        if not isinstance(val, (dict, list)):
            print(f"  {key}: {val}")

    if "logLoss" in metrics:
        print(f"\nLog Loss: {metrics.get('logLoss')}")
    if "auPrc" in metrics:
        print(f"PR AUC: {metrics.get('auPrc')}")
    if "auRoc" in metrics:
        print(f"ROC AUC: {metrics.get('auRoc')}")
else:
    print("Notice: Load a valid 'model' object in Cell 2 to inspect evaluation metrics.")

## 2. Feature Importance / Attributions

Extract global feature attributions to evaluate feature importance across trained models.

In [ ]:
if "model" in locals() and isinstance(model, aiplatform.Model):
    print(f"Fetching global feature attributions for model '{model.resource_name}'...")
    attributions = get_model_feature_attributions(model=model, config=config)
    if attributions:
        print("\n--- Global Feature Importance ---")
        sorted_attributions = sorted(attributions.items(), key=lambda x: x[1], reverse=True)
        for feature, score in sorted_attributions:
            print(f"  {feature:<20}: {score:.4f}")
    else:
        print("No feature attributions available for this model evaluation.")
else:
    print("Notice: Load a valid 'model' object in Cell 2 to inspect feature attributions.")

## 3. Vertex AI Experiment Tracking

List and compare pipeline runs, hyperparameters, and evaluation metrics across experiments.

In [ ]:
print(f"Fetching experiment runs for '{config.experiment_name}'...")
try:
    df = list_experiment_runs(config=config)
    if df is not None and not df.empty:
        print(f"Found {len(df)} experiment run(s):")
        display_cols = [
            c for c in df.columns if any(k in c for k in ["name", "param", "metric", "time"])
        ]
        print(df[display_cols if display_cols else df.columns].head())
    else:
        print(f"No experiment runs logged yet under '{config.experiment_name}'.")
except Exception as e:
    print(f"Notice: Vertex AI Experiment '{config.experiment_name}' not yet initialized: {e}")